Esta notebook contiene bloques de código útiles para realizar Q-learning en el entorno "Continuous Mountain Car"

In [ ]:
import numpy as np
import gymnasium as gym
import random 

In [ ]:
# Cambiar render_mode a rgb_array para entrenar/testear
env_id = 'MountainCarContinuous-v0'
env = gym.make(env_id, render_mode='human')

Observation Space

In [ ]:
env.observation_space

Action Space

In [ ]:
env.action_space

Discretización de los estados

In [ ]:
x_space = np.linspace(-1.2, 0.6, 100)
vel_space = np.linspace(-0.07, 0.07, 100)
x_space

Obtener el estado a partir de la observación

In [ ]:
def get_state(obs):
    x, vel = obs
    x_bin = np.digitize(x, x_space)
    vel_bin = np.digitize(vel, vel_space)
    return x_bin, vel_bin

In [ ]:
state = get_state(np.array([-0.4, 0.02]))
state

Discretización de las acciones

In [ ]:
actions = list(np.linspace(-1, 1, 10))
actions

In [ ]:
def get_sample_action():
    return random.choice(actions)

Inicilización de la tabla Q

In [ ]:
Q = np.zeros((len(x_space) + 1, len(vel_space) + 1, len(actions)))
Q

Obtención de la acción a partir de la tabla Q

In [ ]:
def optimal_policy(state, Q):
    action = actions[np.argmax(Q[state])]
    return action

Epsilon-Greedy Policy

In [ ]:
def epsilon_greedy_policy(state, Q, epsilon=0.1):
    explore = np.random.binomial(1, epsilon)
    if explore:
        action = get_sample_action()
    else:
        action = optimal_policy(state, Q)
        
    return action

Ejemplo de episodio 

In [ ]:
obs,_ = env.reset()
print(obs)
done = False
total_reward = 0
state = get_state(obs)
steps = 0
while not done:
    steps += 1
    # Acción del modelo
    action = epsilon_greedy_policy(state, Q, 0.5)
    
    # Indice de la accion en Q
    action_idx = actions.index(action)
    
    # Acción del ambiente
    real_action = np.array([action])
     
    obs, reward, done, _, _ = env.step(real_action)
    next_state = get_state(obs)
    
   # Usar action_idx para actualizar Q
   # Q[state][action_idx] = ... # Completar
   
   # Actualizar estado
    state = next_state
   
    total_reward += reward

    env.render()

env.close()    
print('total_reward', total_reward)
print('steps', steps)

---
## Sección de Implementación: Q-Learning tabular y Dyna-Q

*Obligatorio IA 2026 — Universidad ORT Uruguay*

Agentes implementados en `q_learning_agent.py` y `dyna_q_agent.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import gymnasium as gym
from q_learning_agent import QLearningAgent
from dyna_q_agent import DynaQAgent

os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models/lost', exist_ok=True)
print('Setup OK')

### 1. Búsqueda de Hiperparámetros

Comparamos 6 configuraciones de discretización variando bins y acciones.
Todos los experimentos usan los mismos hiperparámetros de aprendizaje:
- α = 0.2, γ = 0.99, ε₀ = 1.0, decay = 0.9995, ε_min = 0.05
- Inicialización optimista: **q_init = 1.0**
- 5000 episodios de entrenamiento (excepto bins_20_act15: pre-entrenado 10k)

In [ ]:
env_gs = gym.make('MountainCarContinuous-v0')

TRAIN_KWARGS = dict(
    alpha=0.2, gamma=0.99, epsilon=1.0,
    epsilon_decay=0.9995, epsilon_min=0.05,
    episodes=5000, verbose=False,
)

grid_configs = [
    ('bins_10_act5',  dict(n_pos_bins=10, n_vel_bins=10, n_actions=5)),
    ('bins_10_act10', dict(n_pos_bins=10, n_vel_bins=10, n_actions=10)),
    ('bins_20_act10', dict(n_pos_bins=20, n_vel_bins=20, n_actions=10)),
    ('bins_30_act15', dict(n_pos_bins=30, n_vel_bins=30, n_actions=15)),
    ('bins_30_act20', dict(n_pos_bins=30, n_vel_bins=30, n_actions=20)),
]

grid_results = []

# Cargar modelo pre-entrenado bins_20_act15
print('Loading bins_20_act15...')
ql_pretrained = QLearningAgent.load('../models/lost/qlearning_pos20_vel20_act15.pkl')
res = ql_pretrained.test_agent(env_gs, episodes=20)
grid_results.append({'config': 'bins_20_act15', 'pos_bins': 20, 'n_actions': 15,
                     'state_space': 21*21*15, 'mean_reward': round(res['mean_reward'],2),
                     'success_rate': res['success_rate'], 'episodes': 10000})

# Entrenar configuraciones nuevas
for name, disc in grid_configs:
    print(f'Training {name}...')
    agent = QLearningAgent(**disc, q_init=1.0)
    agent.train_agent(env_gs, **TRAIN_KWARGS)
    agent.save(f'../models/lost/qlearning_{name}.pkl')
    res = agent.test_agent(env_gs, episodes=20)
    n_p, n_v, n_a = disc['n_pos_bins'], disc['n_vel_bins'], disc['n_actions']
    grid_results.append({'config': name, 'pos_bins': n_p, 'n_actions': n_a,
                         'state_space': (n_p+1)*(n_v+1)*n_a,
                         'mean_reward': round(res['mean_reward'],2),
                         'success_rate': res['success_rate'], 'episodes': 5000})
    print(f'  mean={res["mean_reward"]:.2f}, success={res["success_rate"]:.0%}')

env_gs.close()

df_grid = pd.DataFrame(grid_results).sort_values('mean_reward', ascending=False)
print('\n=== RESULTADOS BÚSQUEDA DE HIPERPARÁMETROS ===')
print(df_grid[['config','pos_bins','n_actions','state_space','mean_reward','success_rate']].to_string(index=False))

**Análisis de resultados:**

| Configuración | Ventaja | Desventaja |
|---|---|---|
| 10×10 bins | Pocos estados, aprendizaje rápido | Aliasing severo: estados distintos mapeados al mismo bin |
| 20×20 bins | Balance óptimo representación/eficiencia | — |
| 30×30 bins | Mejor representación | Más estados a explorar, requiere más episodios |
| 5 acciones | Rápido | Discretización gruesa de fuerza, pierde control fino |
| 15 acciones | Buen balance | — |
| 20 acciones | Control preciso | Espacio de acción grande, exploración más costosa |

El espacio de estados crece como O(bins² × actions). La representación útil crece más lentamente — muchos estados nunca se visitan. 20×20 bins con 10-15 acciones tiende a ser el punto óptimo para este ambiente.

### 2. Curvas de Aprendizaje: Q-Learning vs Dyna-Q

Comparamos las curvas de aprendizaje de los dos mejores modelos entrenados.

In [ ]:
ql_m = QLearningAgent.load('../models/lost/qlearning_pos20_vel20_act15.pkl')
dq_m = DynaQAgent.load('../models/lost/dynaq_pos20_vel20_act15.pkl')

ql_r = np.array(ql_m.training_rewards)
dq_r = np.array(dq_m.training_rewards)

def rolling_mean(arr, w=100):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

# Plot 1: Q-Learning
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ql_r, alpha=0.25, color='steelblue', lw=0.5)
ax.plot(np.arange(99, len(ql_r)), rolling_mean(ql_r), color='steelblue', lw=2, label='Media móvil')
ax.axvline(732,  color='red',    ls='--', alpha=0.7, label='Primer éxito (ep 732)')
ax.axvline(2368, color='orange', ls='--', alpha=0.7, label='Media ≥ 50 (ep 2368)')
for x, lbl in [(366,'Exploración'),(1550,'Descubrimiento'),(3184,'Convergencia'),(7000,'Estable')]:
    if x < len(ql_r): ax.axvspan(max(0,x-366), min(len(ql_r),x+366), alpha=0.04, color='gray')
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Q-Learning — Curva de Aprendizaje (q_init=1.0, 20×20, 15 acciones)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/ql_learning_curve.png', dpi=150)
plt.show()
print('Saved ql_learning_curve.png')

# Plot 2: Dyna-Q
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dq_r, alpha=0.25, color='darkorange', lw=0.5)
ax.plot(np.arange(99, len(dq_r)), rolling_mean(dq_r), color='darkorange', lw=2, label='Media móvil')
ax.axvline(371, color='red', ls='--', alpha=0.7, label='Primer éxito (ep 371)')
ax.set_xlabel('Episodio'); ax.set_ylabel('Recompensa')
ax.set_title('Dyna-Q (n=10) — Curva de Aprendizaje (q_init=0, 20×20, 15 acciones)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/dq_learning_curve.png', dpi=150)
plt.show()
print('Saved dq_learning_curve.png')

# Plot 3: Comparison (primeros 5000 eps)
N = min(5000, len(ql_r), len(dq_r))
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.arange(99, N), rolling_mean(ql_r[:N]), color='steelblue',   lw=2,
        label='Q-Learning (10k eps total, q_init=1.0, 95% éxito)')
ax.plot(np.arange(99, N), rolling_mean(dq_r[:N]), color='darkorange', lw=2,
        label='Dyna-Q (5k eps, n_planning=10, 100% éxito)')
ax.axhline(0, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('Episodio (primeros 5000)'); ax.set_ylabel('Media móvil recompensa')
ax.set_title('Q-Learning vs Dyna-Q — Primeros 5000 episodios')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/ql_vs_dq_comparison.png', dpi=150)
plt.show()
print('Saved ql_vs_dq_comparison.png')

### 3. Análisis de Fallas: q_init = 0

**Estructura del reward:** `r = -0.1·a²` por paso + `100` al llegar a la meta `(x ≥ 0.45)`.
Sin la meta, todos los rewards son negativos.

**¿Por qué q_init=0 siempre falla?**

1. Q-table inicializada en 0: `Q(s,a) = 0` para todos los estados no visitados.
2. Primera visita a `(s,a)` con fuerza `|a|>0`: `Q(s,a)` se vuelve negativo (`reward = -0.1·a²`).
3. Con ε decayendo, el agente prefiere estados NO visitados `(Q=0)` sobre acciones con fuerza grande.
4. **Resultado:** el agente aprende fuerza ≈ 0 para minimizar la penalidad.
5. La meta nunca se alcanza → el reward `+100` nunca se propaga.

**Solución: Inicialización Optimista (`q_init=1.0`)**

- `Q=1.0` para estados no visitados > todos los valores negativos visitados.
- El agente **debe** explorar cada estado antes de concluir que es malo.
- Una vez que encuentra la meta, el valor `+100` se propaga hacia atrás.

| Config | epsilon_decay | q_init | Primer éxito | Test success |
|---|---|---|---|---|
| fast decay | 0.995 | 0.0 | nunca | 0% |
| slow decay | 0.9997 | 0.0 | nunca | 0% |
| **slow decay** | **0.9995** | **1.0** | **ep 732** | **95%** |
| Dyna-Q (n=10) | 0.9995 | 0.0 | ep 371 | 100% |

> **Nota Dyna-Q:** el planning (`n=10`) basta para que Dyna-Q encuentre la meta sin inicialización optimista, porque las actualizaciones simuladas propagan el valor de la meta más rápidamente que Q-Learning puro.

### 4. Evaluación Final del Mejor Modelo

In [ ]:
env_eval = gym.make('MountainCarContinuous-v0')
best_agent = QLearningAgent.load('../models/lost/qlearning_pos20_vel20_act15.pkl')
final = best_agent.test_agent(env_eval, episodes=50)
env_eval.close()

print('=' * 45)
print('EVALUACIÓN FINAL (50 episodios)')
print('=' * 45)
print(f'Agente      : Q-Learning tabular')
print(f'Discretizer : 20×20 bins, 15 acciones')
print(f'Hiperparáms : α=0.2, γ=0.99, q_init=1.0')
print(f'Episodios   : 10000')
print()
print(f'Recompensa media : {final["mean_reward"]:.2f} ± {final["std_reward"]:.2f}')
print(f'Tasa de éxito    : {final["success_rate"]:.0%}')